# Analyse Urgences Pédiatriques — Process Mining & Simulation

**Objectif** : explorer l'event log d'urgences, découvrir le vrai parcours patient,
identifier les goulots, et poser les bases d'une simulation.

## Plan
1. Chargement & nettoyage
2. Stats descriptives (arrivées, LOS, modes de sortie)
3. Process mining avec PM4Py (découverte + variantes + bottlenecks)
4. Distributions pour simulation
5. Pistes de scénarios what-if

**Prérequis** : `pip install pandas numpy matplotlib seaborn pm4py`

## 1. Chargement & nettoyage

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

CSV = '2025_11_10_TrackingUrg.csv'

df = pd.read_csv(CSV, sep=';', encoding='utf-8-sig')
# colonnes vides en fin (;;;;)
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()

In [ ]:
# Parsing dates
for c in ['date_arrivée', 'date_sortie', 'loc_heure_debut', 'loc_heure_fin']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# Nettoyage : lignes sans local ou sans timestamp
df = df.dropna(subset=['DOSSIER_ID', 'loc_local_libelle', 'loc_heure_debut'])
df['loc_local_libelle'] = df['loc_local_libelle'].str.strip()

# Durée par étape (min)
df['duree_min'] = (df['loc_heure_fin'] - df['loc_heure_debut']).dt.total_seconds() / 60

# LOS global (min) par dossier
df['los_min'] = (df['date_sortie'] - df['date_arrivée']).dt.total_seconds() / 60

print(f"Dossiers uniques : {df['DOSSIER_ID'].nunique()}")
print(f"Patients uniques : {df['PATIENT_ID'].nunique()}")
print(f"Locaux uniques   : {df['loc_local_libelle'].nunique()}")
print(f"Période          : {df['date_arrivée'].min()} → {df['date_arrivée'].max()}")

## 2. Stats descriptives

In [ ]:
# Mode de sortie
sortie = df.drop_duplicates('DOSSIER_ID')['mode_de_sortie'].value_counts()
print(sortie)
sortie.plot(kind='barh', title='Modes de sortie'); plt.tight_layout(); plt.show()

In [ ]:
# Distribution LOS (clip outliers à 24h pour lisibilité)
los = df.drop_duplicates('DOSSIER_ID')['los_min'].dropna()
print(f"LOS médian : {los.median():.1f} min")
print(f"LOS p90    : {los.quantile(0.9):.1f} min")
los.clip(0, 24*60).hist(bins=60)
plt.xlabel('LOS (min)'); plt.title('Durée de séjour par dossier'); plt.show()

In [ ]:
# Heatmap arrivées heure × jour
arr = df.drop_duplicates('DOSSIER_ID').copy()
arr['heure'] = arr['date_arrivée'].dt.hour
arr['jour']  = arr['date_arrivée'].dt.day_name()
pivot = arr.pivot_table(index='jour', columns='heure', values='DOSSIER_ID', aggfunc='count')
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot.reindex([d for d in order if d in pivot.index])
plt.figure(figsize=(12,4))
sns.heatmap(pivot, cmap='YlOrRd', annot=False)
plt.title('Arrivées par heure × jour'); plt.show()

In [ ]:
# Top locaux par fréquence et par temps moyen passé
stats_local = df.groupby('loc_local_libelle').agg(
    n=('DOSSIER_ID','count'),
    duree_moy=('duree_min','mean'),
    duree_med=('duree_min','median')
).sort_values('n', ascending=False)
stats_local.head(20)

## 3. Process Mining (PM4Py)

In [ ]:
# Préparation event log PM4Py
import pm4py

log_df = df.rename(columns={
    'DOSSIER_ID': 'case:concept:name',
    'loc_local_libelle': 'concept:name',
    'loc_heure_debut': 'time:timestamp'
})[['case:concept:name','concept:name','time:timestamp','loc_heure_fin']].copy()
log_df['case:concept:name'] = log_df['case:concept:name'].astype(str)
log_df = log_df.sort_values(['case:concept:name','time:timestamp'])
log_df.head()

In [ ]:
# Variantes (chemins distincts)
variants = pm4py.get_variants(log_df)
print(f"Nombre de variantes : {len(variants)}")
top = sorted(variants.items(), key=lambda x: -len(x[1]) if isinstance(x[1], list) else -x[1])[:10]
for v, cases in top:
    n = len(cases) if isinstance(cases, list) else cases
    print(f"{n:5d}  {' → '.join(v)[:120]}")

In [ ]:
# Directly-Follows Graph (process map) + export PNG
dfg, start, end = pm4py.discover_dfg(log_df)
pm4py.save_vis_dfg(dfg, start, end, 'dfg_urgences.png')
print('Process map sauvé → dfg_urgences.png')

In [ ]:
# DFG performance (temps moyen entre activités) — identifie les bottlenecks
perf_dfg, s, e = pm4py.discover_performance_dfg(log_df)
pm4py.save_vis_performance_dfg(perf_dfg, s, e, 'dfg_perf_urgences.png')
print('Performance map sauvée → dfg_perf_urgences.png')

## 4. Distributions pour simulation

In [ ]:
# Inter-arrivées (min) par heure de la journée — utile pour simuler un processus non-stationnaire
arrivees = df.drop_duplicates('DOSSIER_ID')[['DOSSIER_ID','date_arrivée']].dropna().sort_values('date_arrivée')
arrivees['inter_min'] = arrivees['date_arrivée'].diff().dt.total_seconds()/60
arrivees['heure'] = arrivees['date_arrivée'].dt.hour
arrivees.groupby('heure')['inter_min'].mean().plot(kind='bar')
plt.ylabel('Inter-arrivée moyenne (min)'); plt.title('Saisonnalité intra-journalière'); plt.show()

In [ ]:
# Matrice de transition entre locaux (probas) — base d'un modèle markovien pour simu
seq = df.sort_values(['DOSSIER_ID','loc_heure_debut']).groupby('DOSSIER_ID')['loc_local_libelle'].apply(list)
pairs = []
for s in seq:
    pairs += list(zip(s, s[1:]))
trans = pd.DataFrame(pairs, columns=['from','to']).value_counts().unstack(fill_value=0)
trans_prob = trans.div(trans.sum(axis=1), axis=0)
trans_prob.round(2).head(10)

## 5. Pistes what-if pour la soutenance

- **Scénario 1 — capacité** : +1 box sur le local le plus saturé (celui avec la file la plus longue en DFG perf).
- **Scénario 2 — pic épidémique** : ×1.5 arrivées sur la tranche 18h–00h → impact sur LOS p90.
- **Scénario 3 — réorganisation IOA** : réduction de 20% du temps en « ATTENTE PED POST-TRI IOA » (staffing renforcé).
- **Scénario 4 — fast-track** : patients « Retour à domicile » routés vers un circuit court dès le tri.

Implémentation rapide avec `SimPy` (Python) ou export des distributions vers **FlexSim**
(arrivées non-stationnaires + durées par local + matrice de transition = modèle calibré).

### Message clé pour l'entretien
> « J'ai reconstruit le parcours réel par process mining, quantifié 3 goulots,
> puis simulé 3 leviers d'action — le plus rentable réduit le LOS p90 de X%. »